# K-Means — shock-response archetypes

Find groups of country-industry-event response profiles without assuming known labels.

**Real data only.** Run ingestion first. Never interpret an empty or skipped model as a finding.

In [ ]:
from pathlib import Path
import sys
for loc in [Path.cwd(),Path.cwd()/'notebooks',Path.cwd()/'research_expansion'/'notebooks']:
    if (loc/'common.py').exists():
        sys.path.insert(0,str(loc.resolve()));break
from common import root, load_panel, load_cases, output

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import pandas as pd
cols=['delta_pp','shock_std_yoy','negative_share','worst_yoy','post_mean_yoy']
df=load_cases().dropna(subset=cols).copy()
if len(df)<10:raise ValueError('Need at least 10 observed cases for clustering')
X=StandardScaler().fit_transform(df[cols]);k=min(3,len(df)-1)
model=KMeans(n_clusters=k,random_state=42,n_init=10).fit(X);df['cluster']=model.labels_
summary=df.groupby('cluster')[cols].mean().reset_index();display(summary)
output('kmeans',{'model':'KMeans','status':'fitted','n_cases':len(df),'k':k,
'silhouette':silhouette_score(X,model.labels_) if 1<len(set(model.labels_))<len(df) else None,
'cluster_summary':summary.to_dict('records'),
'assignments':df[['country','industry_code','event','cluster']].to_dict('records'),
'limitation':'Descriptive segments. Cluster numbers are not inherently ordered risk categories.'})
summary.set_index('cluster')[['delta_pp','shock_std_yoy','worst_yoy','post_mean_yoy']].plot.bar(
    ylabel='Percent or percentage points, by feature',title='Observed cluster profiles')
import matplotlib.pyplot as plt
plt.show()
